### FUNCION PARA POBLAR LA BASE DE DATOS
```python
Inserta los datos de un DataFrame en una colección de MongoDB.

In [1]:
# pip install pymongo pandas
import pandas as pd
from pymongo import MongoClient

cliente = MongoClient("mongodb://localhost:27017/")
db = cliente["ETL_AIRBNB"]

# Carga cada archivo directo desde la URL (pandas lee .gz automáticamente)
urls = {
    "listings": "https://data.insideairbnb.com/argentina/ciudad-aut%C3%B3noma-de-buenos-aires/buenos-aires/2026-01-25/data/listings.csv.gz",
    "reviews":  "https://data.insideairbnb.com/argentina/ciudad-aut%C3%B3noma-de-buenos-aires/buenos-aires/2026-01-25/data/reviews.csv.gz",
    "calendar": "https://data.insideairbnb.com/argentina/ciudad-aut%C3%B3noma-de-buenos-aires/buenos-aires/2026-01-25/data/calendar.csv.gz",
}

for nombre, url in urls.items():
    print(f"Cargando {nombre}...")
    df = pd.read_csv(url, compression='gzip', low_memory=False)
    df = df.where(pd.notnull(df), None)  # convierte NaN a None para MongoDB
    registros = df.to_dict("records")
    db[nombre].drop()  # por si ya existe, para no duplicar
    db[nombre].insert_many(registros)
    print(f"  ✓ {len(registros)} registros insertados en '{nombre}'")

print("Listo!")

Cargando listings...
  ✓ 27348 registros insertados en 'listings'
Cargando reviews...
  ✓ 1042702 registros insertados en 'reviews'
Cargando calendar...
  ✓ 9982072 registros insertados en 'calendar'
Listo!
